# 00 · Why the old SignLens models failed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsemaniJoshua/SignLens/blob/main/notebooks/00_diagnose_old_models.ipynb)

This notebook **reproduces the failure with numbers** using the data already in the repo. Nothing here needs a GPU (about 3 minutes on Colab's free CPU).

It answers four questions:

1. Why did training report high accuracy while the live camera predicted the wrong letters?
2. What happens when the model meets a **new person** (the real-world case)?
3. Does the new preprocessing (hand-only, normalized, geometric augmentation) fix it?
4. Is there a bug in how the API turns model outputs into letters?

A small scikit-learn network (same size as the old Dense model) is used so the comparison is quick and fair. The real training happens in notebooks 02 and 03.

In [ ]:
#@title Setup: get the code and data (run this first)
import os, sys, subprocess

REPO_URL = "https://github.com/AsemaniJoshua/SignLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    REPO_DIR = "/content/SignLens"
    if not os.path.isdir(REPO_DIR):
        # Shallow + sparse clone: only notebooks/ and the small keypoint files
        # (~225 MB, a few seconds). Skips node_modules, videos and images.
        sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {BRANCH} "
           f"--filter=blob:none --sparse {REPO_URL} {REPO_DIR}")
        sh(f"cd {REPO_DIR} && git sparse-checkout set --no-cone /notebooks/ "
           "/model/dataset/static_keypoints_optimized/ "
           "'/model/dataset/video_keypoints_no_face/*/[0-9].npy' "
           "'/model/initial build/dataset/' /model/model_endpoints/app/")
    pass
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SignLens"   # outputs survive the session
else:
    REPO_DIR = os.environ.get("SIGNLENS_REPO_DIR", os.path.abspath(".."))
    WORK_DIR = os.environ.get("SIGNLENS_WORK_DIR", os.path.join(REPO_DIR, "work"))

MODEL_DIR = os.path.join(REPO_DIR, "model")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR = os.path.join(WORK_DIR, "models")
for d in (WORK_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))

import numpy as np
import signlens_core as sc
QUICK = os.environ.get("SIGNLENS_QUICK") == "1"   # tiny settings for automated tests
print(f"signlens_core {sc.__version__} | repo data: {MODEL_DIR} | outputs: {WORK_DIR}")

## 1. The old static pipeline, measured the old way

`train_model_static.py` added noise copies of every sample **and then** shuffled and split. Every test sample therefore had an almost-identical twin in the training set, and the frames themselves are consecutive video frames that look the same. The number this produces measures memorization, not recognition.

In [ ]:
import glob, warnings
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")
rng = np.random.default_rng(0)

def mlp():
    return make_pipeline(StandardScaler(), MLPClassifier((128, 64), alpha=1e-3, max_iter=400,
                                                         early_stopping=True, random_state=0))

# exactly what the old script loaded: the full 258-number Holistic vector
X, y = [], []
for p in sorted(glob.glob(f"{MODEL_DIR}/dataset/static_keypoints_optimized/*/*.npy")):
    X.append(np.load(p)); y.append(os.path.basename(os.path.dirname(p)))
X, y = np.array(X), np.array(y)
Xa = np.concatenate([X, X + rng.normal(0, 0.01, X.shape)]); ya = np.concatenate([y, y])
idx = rng.permutation(len(Xa)); cut = int(0.9 * len(idx))
old = mlp().fit(Xa[idx[:cut]], ya[idx[:cut]])
old_reported = np.mean(old.predict(Xa[idx[cut:]]) == ya[idx[cut:]])
print(f"Accuracy the OLD way (augment, then random split): {old_reported:.1%}")
print(f"Feature vector length: {X.shape[1]}  (132 of these numbers describe the BODY, not the hand)")

## 2. The real test: a different person and camera

The repo actually contains **two different signers** for the same letters:

* **YouTube**: frames cut from one "ASL Manual Alphabet" video (one signer, a front clip and a side clip per letter), in `model/dataset/static_keypoints_optimized`
* **Webcam**: 30 webcam recordings per letter for A–S, in `model/initial build/dataset`. The newer pipeline never used these.

Training on one signer and testing on the other is exactly what happens when a new user opens the app. Three feature types are compared:

| | features |
|---|---|
| **old** | raw MediaPipe hand coordinates (position in the frame, size, left/right as-is) |
| **normalized** | `sc.hand_features`: wrist at origin, palm length = 1, left hands mirrored, aspect ratio corrected, plus joint angles and fingertip distances |
| **normalized + augment** | the same, plus realistic geometric augmentation (finger lengths, viewing angle, tilt, mirror) on training data only |

In [ ]:
yt = sc.load_repo_youtube_frames(MODEL_DIR)
yt = yt.subset(yt.source == "youtube")
cam = sc.load_repo_webcam_sequences(MODEL_DIR, frame_step=3)
shared = [l for l in sc.sort_labels(yt.label) if l in set(cam.label)]
A = yt.subset(np.isin(yt.label, shared)); B = cam.subset(np.isin(cam.label, shared))
print(f"{len(shared)} letters recorded by both signers: {''.join(shared)}  (chance = {1/len(shared):.1%})")

def raw(S):  return S.lm.reshape(len(S), -1)
def norm(S): return sc.batches.static_batch(S.normalized(), range(len(S)))
def norm_aug(S, k=10):
    P = S.normalized(); idx = np.repeat(np.arange(len(S)), k)
    Xa = np.concatenate([sc.batches.static_batch(P, range(len(S))), sc.batches.static_batch(P, idx, rng, augment=True)])
    return Xa, np.concatenate([S.label, S.label[idx]])

results = {}
for name, tr, te in [("YouTube -> webcam", A, B), ("webcam -> YouTube", B, A)]:
    r = {}
    r["old"] = np.mean(mlp().fit(raw(tr), tr.label).predict(raw(te)) == te.label)
    r["normalized"] = np.mean(mlp().fit(norm(tr), tr.label).predict(norm(te)) == te.label)
    Xa, ya = norm_aug(tr, k=10 if not QUICK else 2)
    r["normalized + augment"] = np.mean(mlp().fit(Xa, ya).predict(norm(te)) == te.label)
    results[name] = r
    print(f"train {name:20s}: " + " | ".join(f"{k} {v:.1%}" for k, v in r.items()))

In [ ]:
import matplotlib.pyplot as plt
kinds = list(next(iter(results.values())).keys())
fig, ax = plt.subplots(figsize=(8, 3.6))
w = 0.26
for i, k in enumerate(kinds):
    vals = [results[n][k] * 100 for n in results]
    bars = ax.bar(np.arange(len(results)) + (i - 1) * w, vals, w, label=k)
    ax.bar_label(bars, fmt="%.0f%%", fontsize=9)
ax.axhline(100 / len(shared), color="grey", ls="--", lw=1, label="random guessing")
ax.set_xticks(range(len(results)), [f"train {n}" for n in results])
ax.set_ylabel("accuracy on the OTHER signer (%)"); ax.set_ylim(0, 105)
ax.set_title(f"Old pipeline reported {old_reported:.0%}. On a new signer it is close to guessing.")
ax.legend(fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

**Reading the chart:** with the old features the model is roughly at chance on a new signer. It learned *where the hand was in the frame* and the body pose of one person, not the handshape. Normalizing and augmenting the hand alone gives a large jump using **the same data and the same network**.

The remaining gap (and the weaker webcam → YouTube direction) comes from having only two signers. More signers, from public datasets and your own recordings, is the next lever (notebook 01).

## 3. Same signer, unseen recordings (why your own tests looked fine)

If train and test come from the *same* person, every method looks good. That is why testing with your own hand at your own desk could pass while other users failed.

In [ ]:
camfull = sc.load_repo_webcam_sequences(MODEL_DIR, frame_step=3)
tr, va, te = sc.grouped_split(camfull.label, camfull.group, test_frac=0.2, val_frac=0.0)
sc.check_no_leak(camfull.group, tr, te)
T, E = camfull.subset(tr), camfull.subset(te)
print(f"webcam train recordings -> held-out webcam recordings ({len(set(E.group))} recordings)")
print(f"  old features {np.mean(mlp().fit(raw(T), T.label).predict(raw(E)) == E.label):.1%}")
print(f"  normalized   {np.mean(mlp().fit(norm(T), T.label).predict(norm(E)) == E.label):.1%}")
print("Same-person accuracy says little about a new user. Always report a held-out SIGNER result.")

## 4. The video model's label bug

The API (`model/model_endpoints/app/api_v1/endpoints.py`) turns the model's output number into a letter with a **hard-coded** list `VIDEO_CLASSES`. The model's real class order comes from the folders it was trained on. The two can be compared directly:

In [ ]:
import re, ast
ep = os.path.join(MODEL_DIR, "model_endpoints", "app", "api_v1", "endpoints.py")
api_classes = ast.literal_eval(re.search(r"VIDEO_CLASSES\s*=\s*(\[.*?\])", open(ep).read(), re.S).group(1))
train_root = os.path.join(MODEL_DIR, "dataset", "video_keypoints_no_face")
train_classes = sorted(os.listdir(train_root))   # Windows lists folders in this order
print(f"API label list : {len(api_classes)} names")
print(f"model trained on: {len(train_classes)} classes  (missing: {sorted(set(api_classes) - set(train_classes))})")
print("\nmodel output -> what the API printed")
wrong = 0
for i, true in enumerate(train_classes):
    shown = api_classes[i]
    wrong += shown != true
    if shown != true:
        print(f"  output {i:2d}: model meant {true:7s} -> API said {shown}")
print(f"\n{wrong} of {len(train_classes)} classes were reported as the WRONG sign, even when the model was right.")

## 5. What is really in the dataset

What counts is **distinct recordings** (people × sessions), not files. 30 augmented copies of one clip are still one example.

In [ ]:
allset = sc.StaticSet.concat([sc.load_repo_youtube_frames(MODEL_DIR), sc.load_repo_webcam_sequences(MODEL_DIR)])
allset.summary()
vids = sc.load_repo_video_sequences(MODEL_DIR)
from collections import Counter
vc = Counter(l for _, l, _ in vids)
print("\nVideo sequences per class (originals, not augmented copies):")
print("  " + "  ".join(f"{k}:{v}" for k, v in sorted(vc.items(), key=lambda kv: sc.sort_labels(vc).index(kv[0]))))
missing = [l for l in sc.data.LETTERS if l not in vc]
print("  letters with NO video at all:", missing)

## Summary

| problem | evidence above | fix in the new notebooks |
|---|---|---|
| accuracy measured on near-copies of training data | section 1 vs 2 | grouped split by recording, plus held-out **signer/source** tests |
| model learned position, size and body, not handshape | old features ≈ chance on a new signer | hand-only features normalized for position, size, handedness and aspect ratio |
| pixel-noise augmentation | — | geometric augmentation on the skeleton, training data only |
| API label list didn't match the model | section 4 | `labels.json` saved with every model and loaded by the predictor |
| tiny number of real recordings | section 5 | public datasets + your own recordings (notebook 01) |
| old code uses `mp.solutions`, which current MediaPipe removed | crashes on new installs | MediaPipe Tasks `HandLandmarker` (`signlens_core/extract.py`) |

Next: **01_build_landmark_dataset.ipynb**.